> ⚠️ First, you need to **select the correct environment/kernel** to run the following notebook. Please select the kernel named **"Python (CSBDeep)"** ⚠️

You should have perform preprocessing of data using Notebook #1 [preprocessing.ipynb](preprocessing.ipynb) and have two stack of images corresponding to high and low SNR training paired data.

# 2D-CARE Training

This notebook performs the complete training pipeline for Content-Aware Image Restoration (Weigert et al, 2018), or CARE model, optimizing it for low SNR SMLM image denoising. 

The neural network at the heart of 2D-CARE is a U-Net based on convolutional neural networks (CNNs). Its "U-shaped" structure is designed for image restoration because it allows for the capture of both the global context (shapes) and local details (texture/pixels):

<figure style="text-align: center;">
    <img src="../images/unet.png" width="600" />
    <figcaption style="font-style: italic; color: gray;">
        U-net architecture (Ronneberger et al., 2015).
    </figcaption>
</figure>

The training pipeline is the same as the original paper:

1. **Data Loading & Preprocessing**: Loading the paired ROI stacks generated by the patch extraction pipeline.
2. **Model Configuration**: Setting up the CARE network architecture and hyperparameters (learning rate, epochs, batch size,...).
3. **Training**: Launching the actual training.

## Importing libraries & functions
In the following section, we will load Python librairies and functions required. Run the next cell to load librairies and dependencies:

In [1]:
# Run to load libraries

import os
import sys
import tifffile

import time as time
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from pathlib import Path
from ipywidgets import widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

sys.path.append(os.path.abspath('..'))

from care.data_processing import do_data_processing
from care.training import do_training
from care.data_augmentation import apply_data_augmentation_to_npz
# from care.main import launch_care_training

## Training dataset generation

In this section, we load the two stack of data, and as described [here](https://nbviewer.org/url/csbdeep.bioimagecomputing.com/examples/denoising2D/1_datagen.ipynb), data are processed into a RawData object, required to run with 2D-CARE. 

**For the CARE pipeline to function correctly, the main directory must be named `Training` and contain two subfolders named exactly as follows**:

```text
└─ Training/
    ├─ Low/
    └─ High/
```

**Data must be in stack, no individual frame, and should have the same naming** (*eg: Training/Low/Training.tif and Training/High/Training.tif*)

Below, you can load the two paired stacks and generates 2D-CARE training data:

In [ ]:
# Run to generate training dataset
fc = FileChooser('.')
fc.title = '<b>Select "Training" directory :</b>'
fc.show_only_dirs = True

twindow_low = widgets.IntText(value=3, description='Time window for low images:', style={'description_width': 'initial'})
twindow_high = widgets.IntText(value=3, description='Time window for high images:', style={'description_width': 'initial'})
simu_checkbox = widgets.Checkbox(value=False, description='Simulated data', indent=False)

button_run = widgets.Button(description='Generate Training Data', button_style='success', icon='play')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        if fc.selected is None:
            print("Error : Please select a directory.")
            return
            
        data_path = fc.selected
        path_obj = Path(data_path)
        if path_obj.name != 'Training':
            print(f"Warning : Directory is named '{path_obj.name}' instead of 'Training'.")
            
        model_name = path_obj.parent.name
        base_save_in = str(path_obj).replace(f'/{path_obj.name}', '/models/')
        save_in = f"{base_save_in}{model_name}"
        
        print("="*60)
        print(f"Path read: {data_path}")
        print(f"Model: {model_name}")
        print(f"Saved in: {save_in}")
        print(f"Parameters: Time windows=({twindow_low.value}, {twindow_high.value}), Simulation={simu_checkbox.value}")
        print("="*60)
        
        print("Launching...\n")
        do_data_processing(data_path, save_in, twindow_low.value, twindow_high.value, simu_checkbox.value)
        print("Finished!")

button_run.on_click(on_button_clicked)
ui = widgets.VBox([fc, widgets.VBox([twindow_low, twindow_high]), simu_checkbox, button_run, output])
display(ui)

## Data augmentation (optional)

TO DO

## Launching 2D-CARE Training

TO DO